# 04c — Pre-training real del D10Sformer

**Proyecto:** D10Sformer — MIA305 (UdeSA, 2026)  
**Fase:** 4c — Pre-training MLM puro sobre el corpus combinado (clubes + selecciones)

## Objetivo

1. **Entrenar 5 épocas** sobre los 11.852 docs del corpus combinado (`pretrain.pkl`).
2. Objective: **MLM puro** (BERT-style), 15% de tokens enmascarables.
3. Evaluar contra el corpus val (`val.pkl`, 1.232 docs de selecciones 2023-2024).
4. Persistir **`best.pt`** (mejor por val_loss) para usar en Fase 4d (fine-tuning).
5. Comparar las curvas finales contra el smoke test de Fase 4b.

## Pre-requisito CRÍTICO

**Hardware accelerator: T4 (o L4) GPU**, no CPU.  
`Runtime → Change runtime type → GPU`.  
Si corre en CPU, el run de 925 pasos tarda ~10 horas en vez de ~15 minutos.

---
## 1. Setup + chequeo de GPU

In [ ]:
import sys
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    PROJECT_ROOT = Path('/content/drive/MyDrive/d10sformer-v2')
    DATA_ROOT = Path('/content/drive/MyDrive/d10sformer')
else:
    PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
    DATA_ROOT = PROJECT_ROOT

sys.path.insert(0, str(PROJECT_ROOT / 'src'))
from paths import ensure_paths, print_paths

paths = ensure_paths(project_root=PROJECT_ROOT, data_root=DATA_ROOT)
print_paths(paths)

# Alias legacy usados en notebooks v1
ROOT = paths.project_root
DATA_PROCESSED = paths.data_processed
CORPUS_DIR = paths.corpus_dir
VOCAB_PATH = paths.vocab_path
CKPT_DIR = paths.checkpoints
CHECKPOINTS_V1 = paths.checkpoints_v1
DATA_RAW = paths.data_raw
DATA_INTERIM = paths.data_interim


In [ ]:
import torch
print(f'torch: {torch.__version__}')
print(f'CUDA disponible: {torch.cuda.is_available()}')
if not torch.cuda.is_available():
    raise RuntimeError(
        '❌ NO HAY GPU. Cambiá el runtime: Runtime → Change runtime type → T4 GPU.\n'
        'El pre-training tarda 10h en CPU vs 15min en T4. Abortando.'
    )
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'Memoria GPU: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')
print('✓ GPU OK — listo para entrenar')

In [ ]:
import sys
import json
import pickle
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader

# paths: ROOT ya definido en setup
# paths: sys.path ya configurado

DATA_PROCESSED = paths.data_processed
CORPUS_DIR = paths.corpus_dir
VOCAB_PATH = paths.vocab_path
CKPT_DIR = paths.checkpoints
CKPT_DIR.mkdir(exist_ok=True)

from data.vocabulary import FootballVocab
from data.tokenizer import MatchTokenizer
from data.dataset import MatchDataset
from data.collator import MLMCollator
from models.d10sformer import D10Sformer, D10SformerConfig
from training.trainer import Trainer, TrainerConfig, LossSpec

---
## 2. Cargar corpus + tokenizer + collator

In [ ]:
vocab = FootballVocab.load(VOCAB_PATH)
tokenizer = MatchTokenizer(vocab, max_seq_length=80)

with open(CORPUS_DIR / 'pretrain.pkl', 'rb') as f:
    pretrain_docs = pickle.load(f)
with open(CORPUS_DIR / 'val.pkl', 'rb') as f:
    val_docs = pickle.load(f)

print(f'Corpus pretrain: {len(pretrain_docs):,} docs')
print(f'Corpus val:      {len(val_docs):,} docs')

ds_train = MatchDataset(pretrain_docs, tokenizer)
ds_val   = MatchDataset(val_docs, tokenizer)

collator = MLMCollator(vocab, mlm_probability=0.15, seed=42)

BATCH_SIZE = 64
train_loader = DataLoader(ds_train, batch_size=BATCH_SIZE, shuffle=True,
                          collate_fn=collator, num_workers=0)
val_loader   = DataLoader(ds_val, batch_size=BATCH_SIZE, shuffle=False,
                          collate_fn=collator, num_workers=0)

steps_per_epoch = len(train_loader)
print(f'\nTrain batches/época: {steps_per_epoch}')
print(f'Val batches: {len(val_loader)}')

---
## 3. Configurar Trainer (5 épocas)

In [ ]:
EPOCHS = 5
MAX_STEPS = steps_per_epoch * EPOCHS
print(f'EPOCHS = {EPOCHS}')
print(f'Pasos totales = {MAX_STEPS} ({steps_per_epoch} × {EPOCHS})')

model_config = D10SformerConfig(
    vocab_size=len(vocab),
    d_model=256, num_layers=6, num_heads=8, d_ff=1024,
    max_seq_length=80, num_segments=8,
    dropout=0.1, attention_dropout=0.1,
    pad_token_id=vocab.encode('[PAD]'),
    tie_mlm_weights=True,
)
model = D10Sformer(model_config)
print(f'Modelo: {model.num_parameters():,} parámetros')

trainer_config = TrainerConfig(
    lr=5e-4, weight_decay=0.01, grad_clip_norm=1.0,
    warmup_ratio=0.1,
    max_steps=MAX_STEPS,
    mixed_precision=True,    # AMP en GPU
    log_every=25,
    eval_every=100,          # ~9 evals durante el run
    save_every=250,          # ~3 checkpoints intermedios (para resumir)
    save_best=True,          # ← este es el que vamos a usar en 4d
    output_dir=str(CKPT_DIR),
    run_name='pretrain_5ep',
    seed=42,
)
loss_spec = LossSpec(use_mlm=True)

trainer = Trainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    config=trainer_config,
    loss_spec=loss_spec,
)
print(f'Device: {trainer.device}')
print(f'AMP enabled: {trainer.use_amp}')
print(f'Output dir: {trainer.output_dir}')

---
## 4. Pre-training (5 épocas)

In [ ]:
import time
t0 = time.time()
trainer.train()
elapsed = time.time() - t0
print(f'\n✓ Pre-training completado en {elapsed/60:.1f} min ({MAX_STEPS/elapsed:.1f} step/s)')
print(f'  Best val_loss: {trainer.best_val_loss:.4f}')
print(f'  Best checkpoint: {trainer.output_dir / "best.pt"}')

---
## 5. Curvas de loss / accuracy / lr

In [ ]:
log_path = trainer.log_path
rows = [json.loads(line) for line in log_path.read_text().splitlines() if line.strip()]
train_rows = [r for r in rows if 'phase' not in r]
eval_rows  = [r for r in rows if r.get('phase') == 'eval']

steps_train = [r['step'] for r in train_rows]
loss_train  = [r['loss'] for r in train_rows]
ppl_train   = [r['mlm_perplexity'] for r in train_rows]
acc_train   = [r['mlm_acc'] for r in train_rows]
lr_train    = [r['lr'] for r in train_rows]

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

axes[0, 0].plot(steps_train, loss_train, label='train')
if eval_rows:
    axes[0, 0].plot([r['step'] for r in eval_rows],
                    [r['val_loss'] for r in eval_rows], 'o-', label='val', linewidth=2)
axes[0, 0].axhline(np.log(len(vocab)), linestyle='--', color='red', alpha=0.5,
                   label=f'random = ln({len(vocab)}) = {np.log(len(vocab)):.2f}')
axes[0, 0].set_xlabel('step'); axes[0, 0].set_ylabel('loss'); axes[0, 0].set_title('MLM loss')
axes[0, 0].legend(); axes[0, 0].grid(alpha=0.3)

axes[0, 1].plot(steps_train, ppl_train, label='train')
if eval_rows:
    axes[0, 1].plot([r['step'] for r in eval_rows],
                    [r['val_mlm_perplexity'] for r in eval_rows], 'o-', label='val', linewidth=2)
axes[0, 1].set_xlabel('step'); axes[0, 1].set_ylabel('perplexity')
axes[0, 1].set_title('Perplexity'); axes[0, 1].set_yscale('log')
axes[0, 1].legend(); axes[0, 1].grid(alpha=0.3)

axes[1, 0].plot(steps_train, acc_train, label='train')
if eval_rows:
    axes[1, 0].plot([r['step'] for r in eval_rows],
                    [r['val_mlm_acc'] for r in eval_rows], 'o-', label='val', linewidth=2)
axes[1, 0].set_xlabel('step'); axes[1, 0].set_ylabel('top-1 acc'); axes[1, 0].set_title('MLM accuracy')
axes[1, 0].legend(); axes[1, 0].grid(alpha=0.3)

axes[1, 1].plot(steps_train, lr_train, color='purple')
for ep in range(1, EPOCHS):
    axes[1, 1].axvline(ep * steps_per_epoch, color='gray', linestyle=':', alpha=0.4)
axes[1, 1].set_xlabel('step'); axes[1, 1].set_ylabel('lr'); axes[1, 1].set_title('LR schedule')
axes[1, 1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(trainer.output_dir / 'training_curves.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# Diagnóstico cuantitativo y comparación con smoke
print('=== Diagnóstico de pre-training ===')
print(f'Random baseline (ln(V)):       {np.log(len(vocab)):.4f}')
print(f'Loss inicial    (paso {steps_train[0]:>4}): {loss_train[0]:.4f}  ppl {ppl_train[0]:8.2f}  acc {acc_train[0]:.4f}')
print(f'Loss final      (paso {steps_train[-1]:>4}): {loss_train[-1]:.4f}  ppl {ppl_train[-1]:8.2f}  acc {acc_train[-1]:.4f}')
print(f'Reducción de loss:              {100 * (loss_train[0] - loss_train[-1]) / loss_train[0]:.1f}%')

if eval_rows:
    val_losses = [r['val_loss'] for r in eval_rows]
    val_accs   = [r['val_mlm_acc'] for r in eval_rows]
    print(f'\nVal evolution:')
    print(f'  primer eval: loss={val_losses[0]:.4f}  acc={val_accs[0]:.4f}')
    print(f'  último eval: loss={val_losses[-1]:.4f}  acc={val_accs[-1]:.4f}')
    print(f'  mejor val_loss: {min(val_losses):.4f} (step {eval_rows[val_losses.index(min(val_losses))]["step"]})')
    print(f'  mejor val_acc:  {max(val_accs):.4f} (step {eval_rows[val_accs.index(max(val_accs))]["step"]})')

# Comparación con smoke (si existe)
smoke_log = CKPT_DIR / 'smoke_test_pretrain' / 'metrics.jsonl'
if smoke_log.exists():
    smoke_rows = [json.loads(l) for l in smoke_log.read_text().splitlines() if l.strip()]
    smoke_eval = [r for r in smoke_rows if r.get('phase') == 'eval']
    if smoke_eval:
        smoke_final_val_loss = smoke_eval[-1]['val_loss']
        smoke_final_val_acc  = smoke_eval[-1]['val_mlm_acc']
        print(f'\nComparación vs smoke (200 pasos, subset 1000 docs):')
        print(f'  smoke val_loss = {smoke_final_val_loss:.4f}')
        print(f'  4c    val_loss = {min(val_losses):.4f}  (mejora: {100*(smoke_final_val_loss - min(val_losses)) / smoke_final_val_loss:.1f}%)')
        print(f'  smoke val_acc  = {smoke_final_val_acc:.4f}')
        print(f'  4c    val_acc  = {max(val_accs):.4f}  (mejora: {100*(max(val_accs) - smoke_final_val_acc) / smoke_final_val_acc:.1f}%)')

---
## 6. Listado de checkpoints guardados

In [ ]:
print(f'Checkpoints en {trainer.output_dir}:')
for p in sorted(trainer.output_dir.glob('*.pt')):
    size_mb = p.stat().st_size / 1024 / 1024
    print(f'  {p.name:<25} {size_mb:>6.1f} MB')

print(f'\n✓ Para Fase 4d (fine-tuning) cargamos: {trainer.output_dir / "best.pt"}')

---
## 7. Conclusiones de Fase 4c (llenar al final)

- [ ] Tiempo total de entrenamiento: _____ min
- [ ] Pasos/segundo medios: _____
- [ ] Loss inicial / final: _____ / _____
- [ ] Reducción de loss: _____%
- [ ] Mejor val_loss: _____ (paso _____)
- [ ] Mejor val_acc: _____ (paso _____)
- [ ] Tamaño de `best.pt`: _____ MB
- [ ] ¿Hay señales de overfitting (val_loss empieza a subir)? _____
- [ ] ¿La curva de loss llegó a plateau o sigue bajando? _____

**Next:** Fase 4d — fine-tuning con `LossSpec(use_mlm=True, use_result=True, use_score=True)` sobre `finetune_train.pkl` (8.658 docs, solo selecciones), partiendo del `best.pt` de este pre-train.